# 1) Imports

In [1]:
import pandas as pd

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.pipeline import make_pipeline

## Modelos

from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import Perceptron, LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.linear_model import SGDClassifier
from xgboost import XGBClassifier

# 2) Carregando Datasets

In [ ]:
try:
    df = pd.read_csv('WDICSV.csv')
    df_atributos = pd.read_excel('atributos.xlsx')
except:
    # Caso esteja lendo apenas o arquivo já processado, basta pular para o passo (6)
    df_final = pd.read_csv('dataset_processado.csv')

# 3) Lista de Países e Atributos

In [3]:
df_paises = df[73108:]
lista_paises = df_paises.drop_duplicates(subset=['Country Name'])['Country Name'].to_list()

lista_atributos = df_atributos['Atributos'].to_list()

# 4) Função para montar dataset por ano

In [4]:
def preparar_dados_ano(df, ano, arquivo_classes):
    
    df_p = df[73108:].copy()
    df_classes = pd.read_excel(arquivo_classes)[['Country', 'Classes']]
    
    # Preenchendo NaN com anos anteriores
    lista_anos = df_p.columns.to_list()[4:-3]
    for a in lista_anos[::-1]:
        df_p[str(ano)] = df_p[str(ano)].fillna(df_p[a])
    
    # Selecionando colunas
    df_p = df_p[['Country Name', 'Indicator Name', str(ano)]]
    df_p = df_p[df_p['Indicator Name'].isin(lista_atributos)]
    
    # Pivot
    pv = df_p.pivot_table(str(ano), 'Country Name', 'Indicator Name')
    
    # Merge com classes
    df_merge = pd.merge(pv, df_classes, left_on='Country Name', right_on='Country')
    df_merge = df_merge.drop('Country', axis=1)
    
    # Remover colunas e linhas com muitos NaN
    df_merge = df_merge.dropna(thresh=len(df_merge)-10, axis=1)
    df_merge = df_merge.dropna(thresh=len(df_merge.iloc[0])-10)
    
    # Preencher NaN com média
    df_merge = df_merge.fillna(df_merge.mean())
    
    return df_merge

# 5) Preparar e concatenar dados de 2019 e 2021

In [ ]:
dados_2019 = preparar_dados_ano(df, 2019, 'paises_2019_rating.xlsx')
dados_2021 = preparar_dados_ano(df, 2021, 'paises_2021_rating.xlsx')

df_final = pd.concat([dados_2019, dados_2021]).reset_index(drop=True)
# df_final.to_csv('dataset_processado.csv')

# 6) Separando dados em treinamento e teste

In [8]:
X = df_final.iloc[:, :-1].to_numpy()
y = df_final.iloc[:, -1].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

# 7) Label Encoding

In [9]:
le = LabelEncoder()
y_train = le.fit_transform(y_train)
y_test = le.transform(y_test)

# 8) Função para testar modelos

In [10]:
def avaliar_modelo(nome, modelo, normalizar=True):
    if normalizar:
        pipe = make_pipeline(StandardScaler(), modelo)
    else:
        pipe = modelo
    
    pipe.fit(X_train, y_train)
    acc = pipe.score(X_test, y_test)
    print(f"{nome}: {acc:.4f}")
    return pipe

# 9) Rodando os modelos

In [16]:
avaliar_modelo("Decision Tree", DecisionTreeClassifier(random_state=42), normalizar=False)
avaliar_modelo("Perceptron", Perceptron(random_state=42))
avaliar_modelo("SVM", SVC())
avaliar_modelo("Logistic Regression", LogisticRegression(max_iter=1000))
avaliar_modelo("Random Forest", RandomForestClassifier(n_estimators=1000, max_depth=2, random_state=42), normalizar=False)
avaliar_modelo("Naive Bayes", GaussianNB())
avaliar_modelo("Adaline (SGD)", SGDClassifier(loss='squared_error', max_iter=1000, tol=1e-3, random_state=42))

# XGBoost

xgb = XGBClassifier(
    n_estimators=650,
    max_depth=5,
    learning_rate=0.01,
    subsample=1,
    random_state=42
)

xgb.fit(X_train, y_train)
print("XGBoost:", xgb.score(X_test, y_test))

Decision Tree: 0.6533
Perceptron: 0.6800
SVM: 0.6267
Logistic Regression: 0.6667
Random Forest: 0.7200
Naive Bayes: 0.4800
Adaline (SGD): 0.2000
XGBoost: 0.7333333333333333


# 10) Cross Validation

In [ ]:
# Apresentando o resultado para o melhor modelo que foi o XGBoost
scores = cross_val_score(xgb, X, le.fit_transform(y), cv=5)
print(f"{scores.mean():.2f} acurácia ± {scores.std():.2f}")

0.75 accuracy ± 0.06
